# 體能等級（老師版）

老師對照用，不發給學生。格子跟 `體能等級.ipynb` 一樣：1 資料、2 前處理、3 訓練、4 你的判斷，最後多一格批改用的對照。

在 repo 的 `neural-networks/teacher/` 直接打開就能跑，放進學生專案的 `神經網路/` 實驗夾也能跑。只用 numpy、matplotlib、tensorflow，寫法跟 `神經網路.ipynb` 一樣：`Normalization` 縮放、`Dense` 搭網路、輸出層 softmax。

委託要學生交出的，以及這本的參考值：

| 委託要交的 | 這本的做法與參考值 |
|---|---|
| 用了哪些欄、不合理的數值怎麼處理、留下多少列 | 11 欄全用；只刪不可能的值，13393 列刪 25 列，留下 13368 列 |
| 有沒有因尺度不同而縮放 | 有，`Normalization` 只用訓練資料的平均和標準差 |
| 網路有幾層、每層幾個單元 | 隱藏層 25、15（sigmoid），輸出層 4（softmax） |
| 測試正確率，和沒有隱藏層時的正確率 | 約 0.74；沒有隱藏層約 0.615 |
| 一張看得出年齡怎麼改變等級的圖 | 固定一組中位數成績，年齡 20→64，男女各一張，畫 A–D 四條機率 |

四級人數幾乎一樣（各約 3350），所以亂猜的正確率是 0.25。換一個 `seed` 或切法，正確率大約會在 ±0.02 內晃動，學生做出 0.70–0.76 都算合理。

## 1 資料

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf

tf.get_logger().setLevel("ERROR")

def loadData(filename):
    try:
        here = Path(__vsc_ipynb_file__).resolve().parent
    except NameError:
        here = Path.cwd()
    # 放在學生的 神經網路/ 實驗夾：資料在旁邊的 data/
    # 留在 repo 的 neural-networks/teacher/：資料在套件的 experiment/data/
    folders = [here / "data", here.parent / "src" / "add_neural_networks" / "experiment" / "data"]
    folder = next((f for f in folders if (f / filename).is_file()), folders[0])
    return np.loadtxt(folder / filename, delimiter=",", dtype=str, encoding="utf-8")

# 性別和等級是文字，先全部當文字讀進來
raw = loadData("body_performance.csv")
header, rows = raw[0], raw[1:]
print("筆數", len(rows))
print("欄位", header.tolist())

grades, counts = np.unique(rows[:, -1], return_counts=True)
print("各等級人數", dict(zip(grades.tolist(), counts.tolist())))

# 性別 M→1、F→0，其他欄轉成數字
names = ["年齡", "性別", "身高", "體重", "體脂率", "舒張壓", "收縮壓", "握力", "坐姿體前彎", "仰臥起坐", "立定跳遠"]
sex = (rows[:, 1] == "M").astype(float)
nums = rows[:, [0, 2, 3, 4, 5, 6, 7, 8, 9, 10]].astype(float)
X_all = np.column_stack([nums[:, 0], sex, nums[:, 1:]])
y_all = np.searchsorted(np.array(["A", "B", "C", "D"]), rows[:, -1])   # A→0、B→1、C→2、D→3

for name, column in zip(names, X_all.T):
    print(f"{name:6s} 最小 {column.min():7.1f}  最大 {column.max():7.1f}")

## 2 前處理

只刪「不可能」的值，可能是真的就留下：

| 規則 | 理由 | 刪幾列 |
|---|---|---|
| 舒張壓 < 40、收縮壓 < 70，或舒張壓 ≥ 收縮壓 | 0、6、14 這種血壓活人量不到；下面的數字不會比上面大 | 9 |
| 握力 = 0 | 有來測就握得出力，0 是沒測到 | 3 |
| 立定跳遠 = 0 | 同上 | 10 |
| 坐姿體前彎 > 50 公分 | 185、213 公分，多打一位數 | 2 |
| 體脂率 > 60% | 78.4%，不可能 | 1 |

仰臥起坐 0 次有 125 人，這可能是真的做不起來，保留。身高 125 公分、體重 26 公斤偏極端，但不是不可能，也保留。共刪 25 列，留下 13368 列。

所有欄都用上。各欄尺度差很多（血壓上百、性別 0/1），用實驗本 2.2 的 `Normalization` 縮放，平均和標準差只用訓練資料算。

In [ ]:
AGE, SEX, HEIGHT, WEIGHT, FAT, DIA, SYS, GRIP, BEND, SITUP, JUMP = range(11)
rules = {
    "血壓不合理": (X_all[:, DIA] >= 40) & (X_all[:, SYS] >= 70) & (X_all[:, DIA] < X_all[:, SYS]),
    "握力 0": X_all[:, GRIP] > 0,
    "立定跳遠 0": X_all[:, JUMP] > 0,
    "體前彎 > 50": X_all[:, BEND] <= 50,
    "體脂率 > 60": X_all[:, FAT] <= 60,
}
for name, keep in rules.items():
    print(f"{name}：{int((~keep).sum())} 列")
ok = np.all(list(rules.values()), axis=0)
print("處理前", len(X_all), "列，處理後", int(ok.sum()), "列")

X = X_all[ok]
y = y_all[ok]

# 打亂後 80% 訓練、20% 測試
rng = np.random.default_rng(0)
order = rng.permutation(len(X))
cut = int(len(X) * 0.8)
train, test = order[:cut], order[cut:]

norm_l = tf.keras.layers.Normalization(axis=-1)
norm_l.adapt(X[train])
X_train, y_train = np.asarray(norm_l(X[train])), y[train]
X_test, y_test = np.asarray(norm_l(X[test])), y[test]
print("訓練", X_train.shape, "測試", X_test.shape)

## 3 訓練

輸出層 4 個單元用 softmax，損失用 `SparseCategoricalCrossentropy`，跟實驗本第三部一樣。隱藏層用實驗本教過的 sigmoid，兩層 25、15 個單元。

對照組只有 softmax 輸出層、沒有隱藏層，等於多類別的邏輯回歸。兩個用同一份訓練、測試資料，同樣的學習率和 epoch。

In [ ]:
def build(hidden):
    tf.random.set_seed(0)
    layers = [tf.keras.Input(shape=(X.shape[1],))]
    layers += [tf.keras.layers.Dense(units, activation="sigmoid") for units in hidden]
    layers += [tf.keras.layers.Dense(4, activation="softmax")]
    model = tf.keras.Sequential(layers)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.003),
        loss=tf.keras.losses.SparseCategoricalCrossentropy(),
        metrics=["accuracy"],
    )
    model.fit(X_train, y_train, epochs=60, batch_size=64, verbose=0)
    return model

model = build((25, 15))
_, acc = model.evaluate(X_test, y_test, verbose=0)
print("每層單元數", [layer.units for layer in model.layers], " 測試正確率", round(acc, 3))

flat = build(())
_, flat_acc = flat.evaluate(X_test, y_test, verbose=0)
print("沒有隱藏層    每層單元數", [layer.units for layer in flat.layers], " 測試正確率", round(flat_acc, 3))

In [ ]:
# 固定一組成績（各欄的中位數），只讓年齡從 20 走到 64，男女用同一組成績
base = np.median(X, axis=0)
ages = np.arange(20, 65)
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, s, title in zip(axes, [1.0, 0.0], ["Male", "Female"]):
    grid = np.tile(base, (len(ages), 1))
    grid[:, AGE] = ages
    grid[:, SEX] = s
    prob = model.predict(norm_l(grid), verbose=0)
    for k, grade in enumerate("ABCD"):
        ax.plot(ages, prob[:, k], label=grade)
    ax.set_title(f"{title}, same scores")
    ax.set_xlabel("age")
axes[0].set_ylabel("probability")
axes[0].legend()
plt.show()

print("固定的成績", {name: round(float(v), 1) for name, v in zip(names, base) if name not in ("年齡", "性別")})

<div style="background:#fff6e0;border-left:6px solid #e09b00;padding:12px 16px;border-radius:4px;color:#1a1a1a;">

<h3 style="margin-top:0;color:#1a1a1a;">4 你的判斷（參考答案，學生要自己寫）</h3>
<ol style="color:#1a1a1a;list-style:decimal;padding-left:1.4em;">
<li style="color:#1a1a1a;"><b>不合理的數值你怎麼處理，為什麼？</b><br>血壓 0、6、14，舒張壓比收縮壓高，握力 0、立定跳遠 0，體前彎 185、213 公分，體脂率 78.4%：這些量不出來，是登錄錯誤或沒測到，刪掉，共 25 列。仰臥起坐 0 次（125 人）可能真的做不起來，留下。只有 25 列，刪或不刪正確率差不到 0.01（批改格：不清理約 0.735）。批改重點是學生有沒有先看最小值、最大值，說得出理由；把 125 個仰臥起坐 0 也刪掉不算錯，但要問他為什麼。</li>
<li style="color:#1a1a1a;"><b>同樣的成績，年齡或性別不同時，等級怎麼走？</b><br>男性：同一組中位數成績，20 歲多半是 C、D，30 歲前後 C 最高，40 多歲變成 B，50 歲以後 A 最高。年紀越大，同樣的成績等級越好。原因是等級是跟同年齡、同性別的人比出來的：每個年齡層都差不多兩成是 A，但年紀大的人握力、跳遠普遍比較低，同樣的成績在 60 歲是前段，在 20 歲只是中段。<br>女性：同一組成績幾乎每個年齡都判成 B。這要小心，這組成績（握力 37.9、跳遠 193）對女性很強，4926 位女性裡只有 14 位做得到，模型在這裡是外插，曲線不可信。學生若寫「女生比較難拿 A」要追問：是資料這樣，還是圖畫在沒資料的地方？比較好的做法是女性改用女性的中位數成績再畫一次。</li>
<li style="color:#1a1a1a;"><b>加上隱藏層，正確率差多少？</b><br>約 0.74 對 0.615，差 12 個百分點左右。沒有隱藏層時每一級的分數都只是各欄的加權相加，分界是直的；年齡要「跟同齡的人比」，同一個握力在不同年齡代表不同等級，這種交互作用直的分界畫不出來，隱藏層可以。（批改格：拿掉年齡、性別只剩約 0.63，可見年齡、性別本身很重要；隱藏層只用一層 16 個約 0.71–0.73。這些數字每次跑會差 0.01 左右。）</li>
</ol>

</div>

In [ ]:
# 批改用：學生常見的不同做法，正確率會差多少
def accuracy_of(Xa, ya, hidden):
    rng = np.random.default_rng(0)
    o = rng.permutation(len(Xa))
    c = int(len(Xa) * 0.8)
    n = tf.keras.layers.Normalization(axis=-1)
    n.adapt(Xa[o[:c]])
    tf.random.set_seed(0)
    layers = [tf.keras.Input(shape=(Xa.shape[1],))]
    layers += [tf.keras.layers.Dense(units, activation="sigmoid") for units in hidden]
    layers += [tf.keras.layers.Dense(4, activation="softmax")]
    m = tf.keras.Sequential(layers)
    m.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.003),
              loss=tf.keras.losses.SparseCategoricalCrossentropy(), metrics=["accuracy"])
    m.fit(np.asarray(n(Xa[o[:c]])), ya[o[:c]], epochs=60, batch_size=64, verbose=0)
    return round(m.evaluate(np.asarray(n(Xa[o[c:]])), ya[o[c:]], verbose=0)[1], 3)

print("不清理，直接訓練        ", accuracy_of(X_all, y_all, (25, 15)))
print("清理但不縮放           ", end="")
tf.random.set_seed(0)
raw_model = tf.keras.Sequential([tf.keras.Input(shape=(11,)), tf.keras.layers.Dense(25, activation="sigmoid"),
                                 tf.keras.layers.Dense(15, activation="sigmoid"), tf.keras.layers.Dense(4, activation="softmax")])
raw_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.003),
                  loss=tf.keras.losses.SparseCategoricalCrossentropy(), metrics=["accuracy"])
raw_model.fit(X[train], y_train, epochs=60, batch_size=64, verbose=0)
print(round(raw_model.evaluate(X[test], y_test, verbose=0)[1], 3))
print("拿掉年齡、性別         ", accuracy_of(np.delete(X, [AGE, SEX], axis=1), y, (25, 15)))
print("只有一層隱藏層 16 個   ", accuracy_of(X, y, (16,)))